# Module 03: Online Store & Serving

## What You'll Learn

- How materialization moves features from offline to online store
- Low-latency feature retrieval with `get_online_features()`
- Feature freshness, TTL behaviour in online context
- The Feature Server (HTTP/gRPC endpoint)
- Online store backends (Redis, PostgreSQL)

## Prerequisites

- Completed Modules 01-02
- Feature store applied from Module 02

---

> **🗺️ DATA STRATEGY**: Online serving is how Feast delivers on "Real-Time Decisioning" (Data Scenario #5 in the strategy). Low-latency feature access for real-time AI decisions — credit scoring, fraud detection, recommendation engines.

## Materialization: Offline → Online

**Materialization** is the process of copying the latest feature values from the offline store into the online store for low-latency serving.

```
Offline Store (PostgreSQL/files)      Online Store (Redis/PG)
┌────────────────────────┐      ┌────────────────────────┐
│ Full history (weeks/months)  │      │ Latest values only       │
│ customer_1: [720, 680, 650] │ ───▶ │ customer_1: 700          │
│ customer_2: [550, 580, 600] │      │ customer_2: 600          │
└────────────────────────┘      └────────────────────────┘
      (for training)                    (for inference, <10ms)
```

> **⚠️ GAP**: Default materialization processes features sequentially and does not scale. Ray and Spark backends exist as pluggable compute engines (Module 06) but require manual configuration. The operator only supports CronJob-based materialization via `oc exec` — a fragile pattern for production.
>
> **⚠️ GAP**: No alerts when materialization fails or features go stale beyond TTL. No OOTB monitoring for feature freshness. This is a P0 gap.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.types import Float32, Int64

# Reuse setup from Module 02 (or run Module 02 first)
# Generate sample data if not already present
if not os.path.exists("data/credit_features_timeseries.parquet"):
    print("Please run Module 02 first to generate the data, or running setup...")
    np.random.seed(42)
    records = []
    base_time = datetime(2024, 1, 1)
    for customer_id in range(1, 51):
        credit_score = np.random.randint(600, 800)
        for week in range(52):
            ts = base_time + timedelta(weeks=week)
            credit_score += np.random.randint(-20, 20)
            credit_score = max(300, min(850, credit_score))
            records.append({
                "customer_id": customer_id,
                "event_timestamp": ts,
                "credit_score": credit_score,
                "debt_to_income_ratio": round(np.random.uniform(0.1, 0.9), 3),
                "num_open_accounts": np.random.randint(1, 15),
                "total_credit_utilization": round(np.random.uniform(0.0, 1.0), 3),
            })
    os.makedirs("data", exist_ok=True)
    pd.DataFrame(records).to_parquet("data/credit_features_timeseries.parquet")

# Setup feature store
os.makedirs("feature_repo", exist_ok=True)

customer = Entity(name="customer", join_keys=["customer_id"])
credit_source = FileSource(
    name="credit_timeseries_source",
    path=os.path.abspath("data/credit_features_timeseries.parquet"),
    timestamp_field="event_timestamp",
)
credit_fv = FeatureView(
    name="credit_history",
    entities=[customer],
    ttl=timedelta(weeks=2),
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="debt_to_income_ratio", dtype=Float32),
        Field(name="num_open_accounts", dtype=Int64),
        Field(name="total_credit_utilization", dtype=Float32),
    ],
    source=credit_source,
)

store = FeatureStore(repo_path="feature_repo")
store.apply([customer, credit_source, credit_fv])
print("✅ Feature store ready")

## Running Materialization

In [ ]:
# Materialize features from offline to online store
# This copies the latest values for each entity into the online store
from datetime import datetime, timedelta

start_date = datetime(2024, 1, 1)
end_date = datetime.now()

print(f"Materializing features from {start_date} to {end_date}...")
store.materialize(
    start_date=start_date,
    end_date=end_date,
)
print("✅ Materialization complete")
print("\nThe online store now contains the LATEST value for each customer.")
print("This is what gets served at inference time.")

### Incremental Materialization

In production, you run `materialize_incremental` on a schedule — it only processes new data since the last run.

In [ ]:
# Incremental materialization — only processes new data
store.materialize_incremental(end_date=datetime.now())
print("✅ Incremental materialization complete (only new data processed)")
print("\nOn RHOAI, this runs as a CronJob managed by the operator.")
print("Typical schedule: every 1-6 hours depending on freshness requirements.")

> **📍 RHOAI STATUS**: The operator manages materialization via CronJob. Schedule is configurable in the FeatureStore CR. However, it uses `oc exec` into the Feast pod — a fragile pattern.
>
> **🔮 UPSTREAM**: Feast+Ray integration (Module 06) enables distributed materialization via RayJob CRs. Not yet operator-managed downstream.

## Online Feature Retrieval

`get_online_features()` retrieves the latest materialized values — this is what your model inference endpoint calls.

In [ ]:
# Retrieve online features (latest values for inference)
import time

entity_rows = [{"customer_id": i} for i in [1, 5, 10, 25, 50]]

start = time.time()
online_features = store.get_online_features(
    features=[
        "credit_history:credit_score",
        "credit_history:debt_to_income_ratio",
        "credit_history:num_open_accounts",
    ],
    entity_rows=entity_rows,
).to_dict()
latency_ms = (time.time() - start) * 1000

print(f"Online feature retrieval ({latency_ms:.1f}ms):")
result_df = pd.DataFrame(online_features)
result_df

In [ ]:
print(f"\nLatency: {latency_ms:.1f}ms (local SQLite)")
print("\nTypical production latencies:")
print("  Redis online store: 1-5ms")
print("  PostgreSQL online store: 5-20ms")
print("  SQLite (local): 10-50ms")
print("\n⚠️ Note: Registry metadata resolution accounts for >50% of this time.")
print("   This is a known structural bottleneck (feast-dev/feast#4710).")

> **⚠️ GAP**: Serving latency overhead from registry metadata. Registry metadata resolution accounts for >50% of `get_online_features()` time. This is a structural bottleneck in the serving path (P0 gap). Workaround: registry caching, or use the Feature Server which maintains a warm cache.
>
> **⚠️ GAP**: Hopsworks (commercial competitor) benchmarks show ~10x lower P99 latency in comparable configurations.

## The Feature Server

For production serving, you don't call the Python SDK directly. Instead, the **Feature Server** provides an HTTP/gRPC endpoint that models call.

```bash
# Start feature server locally (for testing)
feast serve --port 6566

# On RHOAI, the operator deploys this automatically
# Access via the service:
# http://feast-feature-server.<namespace>.svc.cluster.local:80
```

### Calling the Feature Server

```python
import requests

# HTTP endpoint for online features
response = requests.post(
    "http://feast-feature-server:80/get-online-features",
    json={
        "features": [
            "credit_history:credit_score",
            "credit_history:debt_to_income_ratio",
        ],
        "entities": {"customer_id": [1, 5, 10]},
    },
)
features = response.json()
```

> **📍 RHOAI STATUS**: The Feature Server is deployed by the operator. It supports both HTTP and gRPC. HA (horizontal scaling) was added in v0.61.0 upstream.
>
> **🔮 UPSTREAM**: v0.61.0 added Feature Server High-Availability on Kubernetes with horizontal scaling support.

## Online Store Backends

| Backend | Latency | Best For | RHOAI Status |
|---------|---------|----------|-------------|
| **Redis** | 1-5ms | High-throughput, low-latency serving | Supported (bring your own Redis) |
| **PostgreSQL** | 5-20ms | Simplicity (dual-use with offline store) | GA (operator-managed) |
| **SQLite** | 10-50ms | Local development only | Available |
| **DynamoDB** | 5-15ms | AWS-native deployments | Community |

### Redis Configuration (Production)

```yaml
# feature_store.yaml with Redis online store
online_store:
  type: redis
  connection_string: redis://feast-redis.my-namespace.svc.cluster.local:6379
  # Or with auth:
  # connection_string: redis://:${REDIS_PASSWORD}@redis-host:6379
```

> **🔮 UPSTREAM**: v0.62.0 added optimized protobuf parsing in Redis online store and parallelized DynamoDB batch reads for improved performance.

## Offline vs Online: Complete Picture

| Aspect | Offline Store | Online Store |
|--------|--------------|-------------|
| **Purpose** | Training data | Inference serving |
| **Data** | Full history | Latest values only |
| **Latency** | Seconds to minutes | Milliseconds |
| **API** | `get_historical_features()` | `get_online_features()` |
| **Query pattern** | Point-in-time joins | Key-value lookup |
| **Backend** | PostgreSQL, Spark, DuckDB | Redis, PostgreSQL |
| **Populated by** | Direct data source access | Materialization |

The key insight: **one feature definition serves both paths**. No training-serving skew.

## 🖥️ UI: Monitoring Feature Freshness

After materialization, you can verify freshness through:

1. **Feast UI** — shows last materialization timestamp per feature view
2. **Prometheus metrics** — `feast_feature_freshness` metric shows time since last materialization
3. **CLI** — `feast feature-views list` shows materialization status

```bash
# Check materialization status via CLI
cd feature_repo && feast feature-views list
```

See [docs/ui-guide.md](../../docs/ui-guide.md) for full UI access instructions.

## Key Takeaways

1. **Materialization** copies latest values from offline to online store
2. **Online store** provides millisecond-latency lookups for inference
3. **Feature Server** is the production serving endpoint (HTTP/gRPC)
4. **Redis** for performance, **PostgreSQL** for simplicity on RHOAI
5. Known gaps: registry latency overhead, no staleness alerting, fragile CronJob materialization

## What's Next

- **Module 04**: Feature Engineering — On-Demand Feature Views, transformations
- **Module 06**: Ray Compute — distributed materialization at scale